## Feature Preparation for Screening Candidates

This notebook generates composition-based features for structural
prediction of candidate materials in the screening space.

### Workflow

1. Load candidate formulas and elemental descriptors.
2. Weight descriptors by stoichiometric coefficients and sum metal contributions.
3. Export 25 input features without target labels.

- Output: `data/compound_new_MCs.csv`

Formulas must contain one or two metals followed by one nonmetal.
Stoichiometric coefficients are used without normalization.
Run cells sequentially.

In [1]:
# ============================================================
# 1. Import dependencies
# ============================================================

import re
import json

import numpy as np
import pandas as pd

In [2]:
# ============================================================
# 2. Load screening candidates and elemental descriptors
# ============================================================

# Select rows with index labels 0 through 25013, inclusive.
tmcs = pd.read_csv(
    "data/screening_space.csv"
).loc[:25013, :]

print("Candidate table shape:", tmcs.shape)

# Specify the element identifier and 13 metal descriptors.
cols_tm = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Nd",
    "Nv",
    "D",
    "EC",
    "TC",
]

# Replace missing metal descriptor values and "/" placeholders with zero.
tm = (
    pd.read_excel(
        "data/6_Structural_Pure.xlsx",
        sheet_name="TM_descriptors",
        usecols=cols_tm,
    )
    .fillna(0, inplace=False)
    .replace("/", 0, inplace=False)
)

# Specify the element identifier and 12 nonmetal descriptors.
cols_c = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "VR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Np",
    "Nv",
    "D",
]

# Replace missing nonmetal descriptor values with zero.
c = (
    pd.read_excel(
        "data/6_Structural_Pure.xlsx",
        sheet_name="C_descriptors",
        usecols=cols_c,
    )
    .fillna(0, inplace=False)
)


Candidate table shape: (25012, 18)


In [3]:
# ============================================================
# 3. Define chemical formula parsing functions
# ============================================================

def parse_compound(compound):
    """Extract element symbols and coefficients from a simple formula.

    Parameters
    ----------
    compound : str
        Chemical formula containing element symbols and optional
        integer or decimal stoichiometric coefficients.

    Returns
    -------
    dict
        Element symbols mapped to their stoichiometric coefficients,
        preserving their order of appearance.

    Notes
    -----
    An omitted coefficient is interpreted as 1.0.
    This parser is intended for simple formulas without parentheses.
    Repeated element symbols overwrite earlier values.
    """
    matches = re.findall(
        r"([A-Z][a-z]*)(\d*\.*\d*)",
        compound,
    )

    result = {}

    for match in matches:
        element, count = match

        # Use a coefficient of 1.0 when no number is provided.
        count = float(count) if count else 1.0
        result[element] = count

    return result


def get_res(compound_name):
    """Return the parsed elemental composition of a compound formula."""
    parsed_result = parse_compound(compound_name)
    return parsed_result


# Check formula parsing using a representative bimetallic nitride.
test = get_res("Mn0.2Cr0.8N")


In [4]:
# ============================================================
# 4. Inspect descriptor columns and the output schema
# ============================================================

display(tm.columns.tolist())
display(c.columns.tolist())

# Define the feature column names for inspection.
# Descriptor order follows the loaded descriptor tables.
columns = (
    [f"AB_{i}" for i in tm.columns.tolist()[1:]]
    + [f"C_{i}" for i in c.columns.tolist()[1:]]
)

display(np.array(columns))

print(
    "Compound column data type:",
    tmcs["compound"].dtype,
)


['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Nd',
 'Nv',
 'D',
 'EC',
 'TC']

['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'VR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Np',
 'Nv',
 'D']

array(['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA',
       'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D'], dtype='<U6')

Compound column data type: object


In [5]:
# ============================================================
# 5. Construct composition-based candidate features
# ============================================================

train_data = []

# Reset row indices before iterating over candidate formulas.
tmcs = tmcs.reset_index(drop=True)

for index, formula in enumerate(tmcs["compound"]):

    compound_feat = [formula]

    composition = get_res(formula)

    keys = list(composition.keys())
    values = list(composition.values())

    # Expected order for a ternary formula:
    # first metal, second metal, and nonmetal.
    if len(keys) > 2:

        A_key = keys[0]
        B_key = keys[1]
        C_key = keys[2]

        A_index = float(values[0])
        B_index = float(values[1])
        C_index = float(values[2])

        # Retrieve the first metal's descriptors, excluding its symbol,
        # and multiply them by its stoichiometric coefficient.
        A_feat = (
            tm[tm["element"] == A_key]
            .values
            .flatten()[1:]
            .astype(float)
            * A_index
        )

        # Retrieve and weight the second metal's descriptors.
        B_feat = (
            tm[tm["element"] == B_key]
            .values
            .flatten()[1:]
            .astype(float)
            * B_index
        )

        # Retrieve and weight the nonmetal descriptors.
        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

        # Sum metal contributions without normalizing their coefficients.
        AB_feat = A_feat + B_feat

    else:

        # Expected order for a binary formula: metal and nonmetal.
        AB_key = keys[0]
        C_key = keys[1]

        AB_index = float(values[0])
        C_index = float(values[1])

        # Weight the single metal's descriptors by its formula coefficient.
        AB_feat = (
            tm[tm["element"] == AB_key]
            .values
            .flatten()[1:]
            .astype(float)
            * AB_index
        )

        # Weight the nonmetal descriptors by its formula coefficient.
        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

    # Concatenate 13 metal features and 12 nonmetal features.
    # Current density and overpotential remain excluded.
    compound_feat += (
        list(AB_feat)
        + list(C_feat)
        # + [
        #     tmcs.loc[index, "CurrentDensity"],
        #     tmcs.loc[index, "overpotential"],
        # ]
    )

    train_data.append(compound_feat)

In [6]:
# ============================================================
# 6. Assemble and export the candidate feature table
# ============================================================

# Use AB_ for the combined metal descriptors and C_ for the nonmetal.
columns = (
    [f"AB_{column}" for column in tm.columns[1:]]
    + [f"C_{column}" for column in c.columns[1:]]
    # + [
    #     "CurrentDensity",
    #     "overpotential",
    # ]
)

colnames = ["compound"] + columns

train_data_ = pd.DataFrame(
    train_data,
    columns=colnames,
)

train_data_.to_csv(
    "data/compound_new_MCs.csv",
    index=False,
)

print("Data shape:", train_data_.shape)

# Exclude only the compound identifier; no target columns are included.
print("Feature number:", train_data_.shape[1] - 1)

display(train_data_.head())


Data shape: (25012, 26)
Feature number: 25


,compound,AB_AN,AB_AW,AB_CR,AB_AR,AB_FIE,AB_SIE,AB_EA,AB_EN,AB_Nd,...,C_CR,C_AR,C_VR,C_FIE,C_SIE,C_EA,C_EN,C_Np,C_Nv,C_D
0,Ag0.95Ce0.05N,47.55,109.48059,1.4535,1.6600,7.47415,20.9215,1.2629,1.8895,9.55,...,0.71,0.56,1.55,14.534,29.6,0.07,3.04,3.0,5.0,0.001251
1,Ag0.9Ce0.1N,48.10,111.09298,1.3770,1.6700,7.37230,20.3930,1.2238,1.8490,9.10,...,0.71,0.56,1.55,14.534,29.6,0.07,3.04,3.0,5.0,0.001251
2,Ag0.8Ce0.2N,49.20,114.31776,1.2240,1.6900,7.16860,19.3360,1.1456,1.7680,8.20,...,0.71,0.56,1.55,14.534,29.6,0.07,3.04,3.0,5.0,0.001251
3,Ag0.5Ce0.5N,52.50,123.99210,0.7650,1.7500,6.55750,16.1650,0.9110,1.5250,5.50,...,0.71,0.56,1.55,14.534,29.6,0.07,3.04,3.0,5.0,0.001251
4,Ag0.95Co0.05N,46.00,105.42145,1.5165,1.6435,7.59125,21.2315,1.2699,1.9275,9.85,...,0.71,0.56,1.55,14.534,29.6,0.07,3.04,3.0,5.0,0.001251


In [7]:
# ============================================================
# 7. Inspect the final output columns
# ============================================================

display(train_data_.columns)

Index(['compound', 'AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE',
       'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D'],
      dtype='object')